# B2-023-generative-models-diffusion — Practice p06 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** generative-adversarial-network

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch
import torch.nn.functional as F

REAL = torch.tensor([[2.0], [-1.0], [0.5], [0.0]], dtype=torch.float64)
FAKE = torch.tensor([[-1.5], [0.3], [0.0], [2.0]], dtype=torch.float64)
ATOL = 1e-10
RTOL = 1e-10

## Solution

Both losses are built only from `F.binary_cross_entropy_with_logits` on raw logits (default mean reduction), so they are numerically stable for large $|\ell|$ and never form a probability that could round to $0$.
A shared guard enforces the `(B,1)` shape contract and equal dtypes.
The discriminator loss is the **sum** of the real-row mean (label $1$) and the fake-row mean (label $0$); the generator loss is the non-saturating mean BCE of the fake logits against label $1$.

In [ ]:
def _check_logits(*tensors):
    for x in tensors:
        if not isinstance(x, torch.Tensor) or x.dim() != 2 or x.shape[-1] != 1:
            raise ValueError("logits must be 2-D tensors with last dimension 1")
        if not torch.is_floating_point(x):
            raise ValueError("logits must be floating point")
    if len({x.dtype for x in tensors}) != 1:
        raise ValueError("logits must share one dtype")


def discriminator_loss(real_logits, fake_logits):
    _check_logits(real_logits, fake_logits)
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def generator_loss(fake_logits):
    _check_logits(fake_logits)
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def saturating_generator_loss(fake_logits):
    """Minimax generator loss: batch mean of log(1 - sigma(l)) (local variant for probe 3)."""
    _check_logits(fake_logits)
    return -F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))


def swapped_discriminator_loss(real_logits, fake_logits):
    """Label-swapped defect: real rows labelled 0, fake rows labelled 1 (probe 5)."""
    _check_logits(real_logits, fake_logits)
    return (F.binary_cross_entropy_with_logits(real_logits, torch.zeros_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits)))

### Probes

Probe 3 compares gradients at $\ell=-6$: the non-saturating gradient is $-(1-\sigma(-6))=-0.997527$, the saturating one is $-\sigma(-6)=-0.002473$, a ratio of $e^{6}\approx403$.

In [ ]:
d1 = discriminator_loss(REAL, FAKE)
g1 = generator_loss(FAKE)

zeros = torch.zeros(8, 1, dtype=torch.float64)
d0, g0 = discriminator_loss(zeros, zeros), generator_loss(zeros)

ell = torch.tensor([[-6.0]], dtype=torch.float64, requires_grad=True)
(grad_ns,) = torch.autograd.grad(generator_loss(ell), ell)
ell_sat = torch.tensor([[-6.0]], dtype=torch.float64, requires_grad=True)
(grad_sat,) = torch.autograd.grad(saturating_generator_loss(ell_sat), ell_sat)

d_stable = discriminator_loss(torch.tensor([[100.0]]), torch.tensor([[-100.0]]))
g_stable = generator_loss(torch.tensor([[-100.0]]))

d_swapped = swapped_discriminator_loss(REAL, FAKE)

bad_shape_errors = []
for call in (lambda: discriminator_loss(torch.zeros(4), torch.zeros(4)),
             lambda: generator_loss(torch.zeros(4))):
    try:
        call()
    except ValueError:
        bad_shape_errors.append(True)
    else:
        bad_shape_errors.append(False)

print(f"probe 1: D {d1.item():.16f}  G {g1.item():.16f}")
print(f"probe 2: D {d0.item():.12f} (2 log 2 = {2 * math.log(2):.12f})  G {g0.item():.12f}")
print(f"probe 3: grad_ns {grad_ns.item():.16f}  grad_sat {grad_sat.item():.16f}  ratio {grad_ns.item() / grad_sat.item():.2f}")
print(f"probe 4: D {d_stable.item():.3e}  G {g_stable.item():.6f}")
print(f"probe 5: swapped D {d_swapped.item():.6f}")
print("probe 6:", bad_shape_errors)

### Answer check

In [ ]:
close = lambda x, v: math.isclose(float(x), v, abs_tol=ATOL, rel_tol=RTOL)
# probe 1
assert close(d1, 1.6208143943388613) and close(g1, 0.7689609285135494)
# 0-dim outputs, float64 dtype on float64 inputs
assert d1.dim() == 0 and g1.dim() == 0 and d0.dim() == 0 and g0.dim() == 0
assert d1.dtype == torch.float64 and g1.dtype == torch.float64
# probe 2
assert close(d0, 2 * math.log(2)) and close(g0, math.log(2))
# probe 3
assert close(grad_ns.item(), -0.9975273768433652)
assert abs(grad_ns.item()) > 100 * abs(grad_sat.item())
assert close(grad_sat.item(), -1.0 / (1.0 + math.exp(6.0)))
# probe 4
assert torch.isfinite(d_stable) and d_stable.item() <= 1e-6
assert d_stable.dtype == torch.float32
assert abs(g_stable.item() - 100.0) <= 1e-4
# probe 5
assert abs(d_swapped.item() - d1.item()) > 0.1
# probe 6
assert bad_shape_errors == [True, True]